### Tokenization
Should produce a token ID. i.e. a number which is then fed into the Embedding Layer.

In [ ]:
import sys
sys.path.insert(0, '..')

from tensorflow import keras
from src.embedding.model import build_model

vocab_size = 1000    # vocabulary size
d = 128              # dimensions we want to work with 64 - 1024
max_seq_length = 20  # max amount of tokens in the input


In [ ]:
%pip install keras
%pip install tensorflow

### Embedding Layer

### Tests

In [ ]:
model = build_model(vocab_size, d, max_seq_length)


##### Model Summary

In [ ]:
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_embedding (Embedding)     │ (None, 20, 128)        │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply_10 (Multiply)          │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_10 (Add)                    │ (None, 20, 128)        │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,000 (500.00 KB)

 Trainable params: 128,000 (500.00 KB)

 Non-trainable params: 0 (0.00 B)

##### Output shape & dtype

In [ ]:
B = 4 # batch size

X = keras.random.randint(shape=(B, max_seq_length), minval=0, maxval=vocab_size, dtype="int32")

Y = model(X)
print("Output shape:", Y.shape)           
print("Output dtype:", Y.dtype)           

Output shape: (4, 20, 128)
Output dtype: <dtype: 'float32'>


##### NaNs / Infs

In [108]:
Y = model(X)
print("Any NaNs:", tf.reduce_any(tf.math.is_nan(Y)).numpy())
print("Any Infs:", tf.reduce_any(tf.math.is_inf(Y)).numpy())
print("Mean abs:", tf.reduce_mean(tf.abs(Y)).numpy())
print("Std:", tf.math.reduce_std(Y).numpy())

Any NaNs: False
Any Infs: False
Mean abs: 0.28496665
Std: 0.32906678


##### Positions matter

In [ ]:
token_id = 42
X_same = keras.ops.full(shape=(1, max_seq_length), fill_value=token_id, dtype="int32") # vector (42, 42.....42)
Y_same = model(X_same) 

diff01 = tf.reduce_mean(tf.abs(Y_same[:, 0, :] - Y_same[:, 1, :])).numpy()
diff12 = tf.reduce_mean(tf.abs(Y_same[:, 1, :] - Y_same[:, 2, :])).numpy()

# expected to be different - same token but different position
print("Mean abs diff pos0 vs pos1:", diff01)
print("Mean abs diff pos1 vs pos2:", diff12)

Mean abs diff pos0 vs pos1: 0.032597102
Mean abs diff pos1 vs pos2: 0.037482776


##### Different tokens, same position

In [128]:
import numpy as np

X_two = np.zeros((1, max_seq_length), dtype="int32")
X_two[0, 0] = 10   # token at pos0
X_two[0, 1] = 999  # token at pos1

Y_two = model(X_two)

# Create another input where pos0 token is different
X_two_b = X_two.copy()
X_two_b[0, 0] = 11

Y_two_b = model(X_two_b)

token_diff = tf.reduce_mean(tf.abs(Y_two[:, 0, :] - Y_two_b[:, 0, :])).numpy()
# expected to be > 0
print("Mean abs diff when token at pos0 changes:", token_diff)

Mean abs diff when token at pos0 changes: 0.36786035
